In [3]:
import os
import numpy as np
import pandas as pd
import cv2
import random
from pyts.image import GramianAngularField
from joblib import Parallel, delayed

import matplotlib.pyplot as plt
from pyts.image import RecurrencePlot

from pyts.image import MarkovTransitionField
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import scipy.signal
import concurrent.futures


from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE


import pickle

from PIL import Image,  ImageFilter, ImageEnhance

## chuyển đổi hình ảnh trên tập dữ liệu CIC DDOS 2019

In [ ]:


# Danh sách các cột được chọn
selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

# Hàm tính ánh xạ vị trí cho từng đặc trưng sử dụng t-SNE
def compute_feature_positions(features, grid_size):
    """
    features: ma trận có kích thước (n_features, n_samples)
    grid_size: kích thước lưới mong muốn (số ô mỗi chiều)
    """
    # Sử dụng t-SNE để ánh xạ các đặc trưng xuống không gian 2 chiều
    tsne = TSNE(n_components=2, random_state=42)
    # Lưu ý: các hàng của features tương ứng với từng đặc trưng
    tsne_results = tsne.fit_transform(features)
    
    # Chuẩn hóa kết quả t-SNE về khoảng [0, 1]
    tsne_min = tsne_results.min(axis=0)
    tsne_max = tsne_results.max(axis=0)
    tsne_norm = (tsne_results - tsne_min) / (tsne_max - tsne_min)
    
    # Ánh xạ các giá trị liên tục sang chỉ số lưới [0, grid_size-1]
    grid_positions = np.floor(tsne_norm * (grid_size - 1)).astype(int)
    return grid_positions

# Tiền xử lý dữ liệu và tính toán ánh xạ đặc trưng (được tính một lần cho toàn bộ dữ liệu)
def prepare_data(input_csv, max_per_label=200):
    df = pd.read_csv(input_csv)
    df = df[selected_columns]
    
    labels = df['Label']
    data = df.drop(columns=['Label'])
    
    # Thay thế giá trị vô hạn và NaN
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)
    
    # Biến đổi dữ liệu (sử dụng log1p để giảm độ lệch)
    data = np.log1p(data + 1)
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)
    
    # Chuẩn hóa dữ liệu
    scaler = MinMaxScaler()
    features = scaler.fit_transform(data.values)
    
    return features, labels

# Tính toán ánh xạ các đặc trưng vào lưới 2D dựa trên t-SNE
def compute_grid_mapping(features, grid_size):
    # Tính ánh xạ dựa trên đặc trưng: với mỗi cột là một đặc trưng, sử dụng một tập con các mẫu

    '''Để giảm thời gian tính toán, ta có thể dùng 100 mẫu ngẫu nhiên nếu số mẫu quá lớn.'''
    # n_samples = features.shape[0]
    # sample_indices = np.random.choice(n_samples, size=min(100, n_samples), replace=False)
    # # Transpose: ma trận (n_features, số mẫu được chọn)
    # feature_matrix = features[sample_indices, :].T  

    ''' chọn toàn bộ tập dữ liệu '''
    feature_matrix = features.T  

    # feature_matrix.shape = (n_features, số mẫu được chọn)
    grid_positions = compute_feature_positions(feature_matrix, grid_size)

    return grid_positions

# Hàm xử lý một task: chuyển đổi một mẫu dữ liệu thành ảnh dựa trên ánh xạ đặc trưng
def process_task(task, grid_positions, grid_size, image_size=224):
    row, label, img_number, output_base = task
    n_features = row.shape[0]
    
    # Tạo lưới rỗng để điền giá trị của các đặc trưng
    grid = np.zeros((grid_size, grid_size), dtype=float)
    count = np.zeros((grid_size, grid_size), dtype=int)
    
    # Gán giá trị của từng đặc trưng vào ô lưới theo ánh xạ đã tính được
    for i in range(n_features):
        x, y = grid_positions[i]
        grid[x, y] += row[i]
        count[x, y] += 1

    # Nếu có ô có nhiều hơn 1 đặc trưng, ta lấy trung bình
    nonzero = count > 0
    grid[nonzero] = grid[nonzero] / count[nonzero]
    
    # Chuẩn hóa lại lưới về khoảng [0, 255]
    grid_norm = ((grid - grid.min()) / (grid.max() - grid.min() + 1e-8)) * 255
   
    
    # Phóng đại ảnh sử dụng phép nhân Kronecker
    s = grid_size  # kích thước lưới ban đầu
    size = image_size  # kích thước ảnh mong muốn
    upscale_factor = size // s  # hệ số phóng đại
    expanded_image = np.kron(grid_norm, np.ones((upscale_factor, upscale_factor)))
    
    scaled_image = expanded_image.astype(np.uint8)
    
    # Phóng to ảnh lên kích thước mong muốn
    img = Image.fromarray(scaled_image, mode='L').convert("RGB")
    img = img.resize((image_size, image_size))
    
    # Data Augmentation
    # 1. Blur
    if random.random() < 0.1:
        img = img.filter(ImageFilter.GaussianBlur(radius=1))

    # 2. Noise
    # if random.random() < 0.1:
    #     img_array = np.array(img)
    #     noise = np.random.randint(-25, 25, img_array.shape, dtype=np.int32)
    #     img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    #     img = Image.fromarray(img_array)

    # # 3. Rotation
    # if random.random() < 0.1:
    #     angle = random.uniform(-15, 15)
    #     img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=(0, 0, 0))

    # # 4. Flip
    # if random.random() < 0.1:
    #     img = img.transpose(Image.FLIP_LEFT_RIGHT) if random.choice([True, False]) else img.transpose(Image.FLIP_TOP_BOTTOM)
    
    # Lưu ảnh
    os.makedirs(os.path.join(output_base, str(label)), exist_ok=True)
    img.save(os.path.join(output_base, str(label), f'{img_number}.png'))

# Hàm chính để chuyển đổi dữ liệu CSV thành tập ảnh sử dụng DeepInsight cải tiến
def deepinsight_conversion(input_csv, output_dir='output_images', max_per_label=200, workers=3, grid_size=32, image_size=224):
    features, labels = prepare_data(input_csv, max_per_label)
    
    # Tính ánh xạ các đặc trưng (số đặc trưng = số cột dữ liệu)
    n_features = features.shape[1]
    # Nếu grid_size chưa đủ lớn, đảm bảo grid_size >= ceil(sqrt(n_features))
    min_grid = int(np.ceil(np.sqrt(n_features)))
    grid_size = max(grid_size, min_grid)
    
    grid_positions = compute_grid_mapping(features, grid_size)  # mảng shape (n_features, 2)
    
    # Tạo thư mục đầu ra cho từng nhãn
    unique_labels = pd.unique(labels)
    for label in unique_labels:
        os.makedirs(os.path.join(output_dir, str(label)), exist_ok=True)
    
    tasks = []
    for label in unique_labels:
        mask = (labels == label).values
        label_features = features[mask][:max_per_label]
        for img_number, row in enumerate(label_features):
            tasks.append((row, label, img_number, output_dir))
    
    # Xử lý song song các task
    def task_wrapper(task):
        process_task(task, grid_positions, grid_size, image_size)
    
    with ThreadPoolExecutor(max_workers=workers) as executor:
        executor.map(task_wrapper, tasks)

# Ví dụ sử dụng hàm:
deepinsight_conversion(
    input_csv='data/csv/cicddos_2019_4_labels.csv',
    output_dir='data/images',
    max_per_label=5000,
    workers=2,
    grid_size=7,  # Bạn có thể điều chỉnh kích thước lưới theo số lượng đặc trưng
    image_size=224
)


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
